# 00 — One-time Setup

Run this notebook **once** (per Google account + Drive folder) before using the `01_run_mlp3.ipynb` or `02_run_mlp3_fixed.ipynb` runners.

**What it does:**
1. Mounts Google Drive (expected account: `hert7739@ox.ac.uk`)
2. Creates `results_pcqm4m_subset/` and `datasets/` folders on Drive
3. Clones the GitHub repo into the VM
4. Installs dependencies
5. Symlinks results + datasets to Drive
6. Logs into WandB
7. Downloads + processes the PCQM4Mv2 dataset onto Drive (~30 min)

**Why separate?** The dataset processing step writes 7.7 GB to Drive. You pay that cost **once** — after this notebook finishes, the run notebooks can spin up in ~5 min.

**Runtime:** GPU is not strictly needed for setup, but use GPU anyway so the dataset loader sanity-check verifies CUDA works.

## 1. Configuration

In [ ]:
# =========================================================
# EDIT IF NEEDED
# =========================================================
GITHUB_REPO     = "https://github.com/mark-znidar/gdl__2.git"
DRIVE_WORKSPACE = "/content/drive/MyDrive"   # where results_pcqm4m_subset/ + datasets/ will live
# =========================================================

DRIVE_RESULTS  = f"{DRIVE_WORKSPACE}/results_pcqm4m_subset"
DRIVE_DATASETS = f"{DRIVE_WORKSPACE}/datasets"
print("Drive results dir: ", DRIVE_RESULTS)
print("Drive datasets dir:", DRIVE_DATASETS)

## 2. Mount Google Drive

Sign in with **hert7739@ox.ac.uk** when the popup appears.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Create Drive folders

In [ ]:
import os
os.makedirs(DRIVE_RESULTS,  exist_ok=True)
os.makedirs(DRIVE_DATASETS, exist_ok=True)
!ls -la {DRIVE_WORKSPACE} | head -20

## 4. Clone the repo

In [ ]:
%cd /content
if not os.path.isdir("gdl__2"):
    !git clone {GITHUB_REPO}
%cd gdl__2
!pwd && git log -1 --oneline

## 5. Install dependencies (~3 min)

In [ ]:
!bash run_colab/setup.sh

## 6. Symlink `results_pcqm4m_subset/` and `datasets/` to Drive

After this, anything written into these dirs by the training code lands directly on Drive.

In [ ]:
!rm -rf results_pcqm4m_subset datasets
!ln -s {DRIVE_RESULTS}  results_pcqm4m_subset
!ln -s {DRIVE_DATASETS} datasets
!ls -la results_pcqm4m_subset datasets

## 7. Log in to WandB

API key at [wandb.ai/authorize](https://wandb.ai/authorize).

In [ ]:
import wandb
wandb.login()

## 8. Download + process the dataset (one-time, ~30 min)

OGB downloads `data.csv.gz` (~52 MB) and builds `geometric_data_processed.pt` (~7.7 GB) directly on Drive via the symlink.

**Coffee break.** Go do something else for half an hour.

In [ ]:
from ogb.lsc import PygPCQM4Mv2Dataset
ds = PygPCQM4Mv2Dataset(root='datasets')
print("Dataset ready:", len(ds), "graphs")
split = ds.get_idx_split()
print({k: len(v) for k, v in split.items()})

## 9. Verify Drive contents

In [ ]:
!du -sh {DRIVE_DATASETS}/pcqm4m-v2/processed/* 2>/dev/null
!ls -la {DRIVE_RESULTS}

## ✅ Done

Now open one of:
- **`01_run_mlp3.ipynb`** — trains L-HKS K=16 MLP3 (learned diffusion times)
- **`02_run_mlp3_fixed.ipynb`** — trains L-HKS K=16 MLP3 (frozen diffusion times)

Each of those will mount the same Drive and re-use the dataset you just processed.